In [1]:
import cv2
import numpy as np

def encoder_dic(valid_data):
  data_dic={}
  (valid_boxes,valid_labels,valid_scores)=valid_data
  for box, label,score in zip(valid_boxes,valid_labels,valid_scores):
    if label not in data_dic:
      data_dic[label]=[[score,box,'kept']]
    else:
      data_dic[label].append([score,box,'kept'])
      
  return data_dic
dic=encoder_dic(valid_data)


def decode_box_coor(box):
  return (box.xmin, box.ymin,box.xmax, box.ymax )

def iou(box1, box2):
  (box1_x1, box1_y1, box1_x2, box1_y2) = decode_box_coor(box1)
  (box2_x1, box2_y1, box2_x2, box2_y2) = decode_box_coor(box2)

  xi1 = max(box1_x1,box2_x1)
  yi1 = max(box1_y1,box2_y1)
  xi2 = min(box1_x2,box2_x2)
  yi2 = min(box1_y2,box2_y2)
  inter_width = xi2-xi1
  inter_height = yi2-yi1
  inter_area = max(inter_height,0)*max(inter_width,0)

  box1_area = (box1_x2-box1_x1)*(box1_y2-box1_y1)
  box2_area = (box2_x2-box2_x1)*(box2_y2-box2_y1)
  union_area = box1_area+box2_area-inter_area 

  iou = inter_area/union_area
  
  return iou

def do_nms(data_dic, nms_thresh):
  final_boxes,final_scores,final_labels=list(),list(),list()
  for label in data_dic:
    scores_boxes=sorted(data_dic[label],reverse=True)
    for i in range(len(scores_boxes)):
      if scores_boxes[i][2]=='removed': continue
      for j in range(i+1,len(scores_boxes)):
        if iou(scores_boxes[i][1],scores_boxes[j][1]) >= nms_thresh:
          scores_boxes[j][2]="removed"

    for e in scores_boxes:
      print(label+' '+str(e[0]) + " status: "+ e[2])
      if e[2]=='kept':
        final_boxes.append(e[1])
        final_labels.append(label)
        final_scores.append(e[0])
    

  return (final_boxes,final_labels,final_scores)



# Laden der YOLO-Modelldateien
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

# Definieren der Klassen, die von YOLO erkannt werden sollen
classes = []
with open("coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Definieren der Eingangsgröße des Bildes
input_size = 416

# Definieren von Farben für jede Klasse
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# Laden des Bildes
img = cv2.imread("personen.jpeg")

# Skalieren des Bildes und Konvertieren in ein Blob
blob = cv2.dnn.blobFromImage(img, 1/255.0, (input_size, input_size), swapRB=True, crop=False)

# Setzen des Blob als Eingang des YOLO-Modells und Durchführen der Vorhersage
net.setInput(blob)
outputs = net.forward(net.getUnconnectedOutLayersNames())

# Durchlaufen der Vorhersagen und Zeichnen von Rechtecken um erkannte Personen
for output in outputs:
    for detection in output:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if class_id == 0 and confidence > 0.5:
            center_x = int(detection[0] * img.shape[1])
            center_y = int(detection[1] * img.shape[0])
            w = int(detection[2] * img.shape[1])
            h = int(detection[3] * img.shape[0])

            x = int(center_x - w/2)
            y = int(center_y - h/2)

            cv2.rectangle(img, (x, y), (x + w, y + h), colors[class_id], 2)
            
            final_data=do_nms(dic, 0.7)
            draw_boxes(frame,final_data)



# Anzeigen des Ergebnisbildes
cv2.imshow("Personendetektion mit YOLO", img)
cv2.waitKey(0)


NameError: name 'valid_data' is not defined